In [1]:
import pandas as pd
from pathlib import Path


def add_rolling_features(
    df,
    windows=(6, 24, 72),
    ops=("mean", "sum"),
    positive_only_for_degree=False,
    targ_prefixes_temp=("temp", "fc_temp"),
    targ_prefixes_prec=("prec", "fc_prec"),
    process_temp=True,
    process_prec=True,
    min_periods=1,
):
    out = df.copy()
    added = []

    def match_prefixes(cols, prefixes):
        return [c for c in cols if any(c.startswith(p) for p in prefixes)]

    cols = df.columns.tolist()
    temp_cols = match_prefixes(cols, targ_prefixes_temp) if process_temp else []
    prec_cols = match_prefixes(cols, targ_prefixes_prec) if process_prec else []

    for col in temp_cols + prec_cols:
        series = df[col]
        is_degree_col = "degree" in col.lower()
        for w in windows:
            rol = series.rolling(window=w, min_periods=min_periods)
            if "mean" in ops:
                newname = f"{col}_roll_mean_{w}"
                out[newname] = rol.mean()
                added.append(newname)
            if "sum" in ops:
                if positive_only_for_degree and is_degree_col:
                    pos_series = series.where(series > 0, 0)
                    newname_pos = f"{col}_roll_sumpos_{w}"
                    out[newname_pos] = pos_series.rolling(window=w, min_periods=min_periods).sum()
                    added.append(newname_pos)
                    newname = f"{col}_roll_sum_{w}"
                    out[newname] = rol.sum()
                    added.append(newname)
                else:
                    newname = f"{col}_roll_sum_{w}"
                    out[newname] = rol.sum()
                    added.append(newname)
    return out, added



In [2]:
# Change this path to your CSV file
csv_path = Path(r"data\massa_2.csv")
date_col = "date"

df = pd.read_csv(csv_path)
df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values(date_col).reset_index(drop=True).set_index(date_col)

# Example usage
windows = [6, 24, 72]
ops = ("mean", "sum")
df_new, added_features = add_rolling_features(
    df,
    windows=windows,
    ops=ops,
    positive_only_for_degree=True,
)

# Save updated CSV
out_csv = csv_path.with_name(csv_path.stem + "_with_rollings.csv")
df_new.reset_index().to_csv(out_csv, index=False)

# Print added features in requested format
for i, feat in enumerate(added_features, 1):
    print(f"- Feature added {i}: {feat}")

C:\Users\Kohler\AppData\Local\Temp\ipykernel_10144\3561641578.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[newname] = rol.mean()
C:\Users\Kohler\AppData\Local\Temp\ipykernel_10144\3561641578.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[newname] = rol.sum()
C:\Users\Kohler\AppData\Local\Temp\ipykernel_10144\3561641578.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once 

- Feature added 1: temp_jungfraujoch_roll_mean_6
- Feature added 2: temp_jungfraujoch_roll_sum_6
- Feature added 3: temp_jungfraujoch_roll_mean_24
- Feature added 4: temp_jungfraujoch_roll_sum_24
- Feature added 5: temp_jungfraujoch_roll_mean_72
- Feature added 6: temp_jungfraujoch_roll_sum_72
- Feature added 7: temp_binn_roll_mean_6
- Feature added 8: temp_binn_roll_sum_6
- Feature added 9: temp_binn_roll_mean_24
- Feature added 10: temp_binn_roll_sum_24
- Feature added 11: temp_binn_roll_mean_72
- Feature added 12: temp_binn_roll_sum_72
- Feature added 13: temp_eggishorn_roll_mean_6
- Feature added 14: temp_eggishorn_roll_sum_6
- Feature added 15: temp_eggishorn_roll_mean_24
- Feature added 16: temp_eggishorn_roll_sum_24
- Feature added 17: temp_eggishorn_roll_mean_72
- Feature added 18: temp_eggishorn_roll_sum_72
- Feature added 19: temp_visp_roll_mean_6
- Feature added 20: temp_visp_roll_sum_6
- Feature added 21: temp_visp_roll_mean_24
- Feature added 22: temp_visp_roll_sum_24
- Fe